<h4>-----------------------------------------------------------------------------<br>Copyright (c) 2024, Lucid Vision Labs, Inc.</h4>
<h5> THE  SOFTWARE  IS  PROVIDED  "AS IS",  WITHOUT  WARRANTY  OF  ANY  KIND,<br>EXPRESS  OR  IMPLIED,  INCLUDING  BUT  NOT  LIMITED  TO  THE  WARRANTIES<br>OF  MERCHANTABILITY,  FITNESS  FOR  A  PARTICULAR  PURPOSE  AND<br>NONINFRINGEMENT.  IN  NO  EVENT  SHALL  THE  AUTHORS  OR  COPYRIGHT  HOLDERS<br>BE  LIABLE  FOR  ANY  CLAIM,  DAMAGES  OR  OTHER  LIABILITY,  WHETHER  IN  AN<br>ACTION  OF  CONTRACT,  TORT  OR  OTHERWISE,  ARISING  FROM,  OUT  OF  OR  IN<br>CONNECTION  WITH  THE  SOFTWARE  OR  THE  USE  OR  OTHER  DEALINGS  IN <br> THE  SOFTWARE.<br>-----------------------------------------------------------------------------</h5>

In [1]:
import time
from datetime import datetime

from arena_api.enums import PixelFormat
from arena_api import enums as _enums
from arena_api.__future__.save import Writer
from arena_api.system import system
from arena_api.buffer import BufferFactory

#### Save: Jpeg
>This example introduces saving JPEG image data in the saving library. It
   shows the construction of image writer, and saves a single JPEG image with 
   configuration parameters.

In [2]:
TAB1 = "  "
pixel_format = PixelFormat.BGR8

In [ ]:
"""
This function waits for the user to connect a device before raising
an exception
"""
tries = 0
tries_max = 6
sleep_time_secs = 10
while tries < tries_max:  # Wait for device for 60 seconds
    devices = system.create_device()
    if not devices:
        print(
            f'Try {tries+1} of {tries_max}: waiting for {sleep_time_secs} '
            f'secs for a device to be connected!')
        for sec_count in range(sleep_time_secs):
            time.sleep(1)
            print(f'{sec_count + 1 } seconds passed ',
                  '.' * sec_count, end='\r')
        tries += 1
    else:
        print(f'Created {len(devices)} device(s)\n')
        break
else:
    raise Exception(f'No device found! Please connect a device and run '
                    f'the example again.')

device = system.select_device(devices)
print(f'Device used in the example:\n\t{device}')

In [4]:
"""
Setup stream values
"""
tl_stream_nodemap = device.tl_stream_nodemap
tl_stream_nodemap['StreamAutoNegotiatePacketSize'].value = True
tl_stream_nodemap['StreamPacketResendEnable'].value = True

In [5]:
device.start_stream()
buffer = device.get_buffer()

#### demonstrates saving a JPEG image
1. converts image to a displayable pixel format
2. prepares image parameters
3. prepares image writer
4. saves image with configuration parameters
5. destroys converted image

#### Convert image
> Convert the image to a displayable pixel format. It is worth keeping in mind the best pixel and file formats for your application. This example converts the image so that it is displayable by the operating system.

In [ ]:
converted = BufferFactory.convert(buffer, pixel_format)
print(f"{TAB1}Converted image to {pixel_format.name}")

#### Prepare image writer
> When saving as .jpg file, the writer optionally can take quality, 
progressive, subsamplng, and optimize as configuration parameters of image(s) 
it would save. If these arguments are not passed at run time, the Writer.save() 
function will configure the writer to defualt quality, progressive, subsamplng,
and optimize.

In [ ]:
print(f'{TAB1}Prepare Image Writer')
writer = Writer.from_buffer(converted)
writer.pattern = 'images/image_<count>.jpg'


#### Save converted buffer with configuration parameters

In [ ]:
''' 
Save function for .jpg file
    buffer :
        buffer to save.
    kwargs (optional args) ignored if not applicable to an .jpg image:
        - 'quality', default is 75.
            Image quality (1 lowest, 100 highest)
        - 'progressive', default is False.
            If true, saves progressive
            Otherwise, saves baseline
        - 'subsampling', default is SC_NO_JPEG_SUBSAMPLING.
            The chroma subsampling to apply
        - 'optimize', default is False
            If true, calculates optimal Huffman coding tables
            Otherwise, does not
'''
writer.save(converted, quality=75, progressive=False, subsampling=_enums.ScJpegSubsamplingList.SC_NO_JPEG_SUBSAMPLING, optimize=False)
print(f'{TAB1}Image saved')

#### Destroy converted buffer to avoid memory leaks

In [9]:
BufferFactory.destroy(converted)

device.requeue_buffer(buffer)

#### Clean up

In [10]:
device.stop_stream()

# Destroy Device
system.destroy_device()